# Creating an Electronic Brochure from Website Links

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-4o-mini'
openai = OpenAI()

API key looks good so far


In [3]:
# A class to represent a Webpage - we first saw this on Day 1

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [5]:
ken = Website("https://roarmarketingconcepts.github.io/")
print(ken.links)

['#intro', '#work', '#resume', '#social', 'https://github.com/ROARMarketingConcepts/AWS-Training/blob/main/AWS_Sagemaker_Models/PCA%20%26%20XGBoost%20Classification/cardiovascular_disease_prediction_notebook.ipynb', 'https://github.com/ROARMarketingConcepts/Recurrent-Neural-Network-Examples/blob/master/RNN_Training_on_Time_Series_Data_Example.ipynb', 'https://github.com/ROARMarketingConcepts/Marketing-Attribution-Using-Markov-Chains/blob/main/Marketing_Channel_Attribution_Using_Markov_Chains.ipynb', 'http://www.screencast.com/t/52LICFT7rxY', 'https://github.com/ROARMarketingConcepts/', 'https://rpubs.com/ROARMarketingConcepts/', 'https://drive.google.com/file/d/1so0HcugmQyg5EvGISEYzyeKD5zEsMwFa/view?usp=share_link', 'https://www.linkedin.com/in/kenwood-roarmarketingconcepts/', 'https://github.com/ROARMarketingConcepts', 'https://rpubs.com/ROARMarketingConcepts', 'mailto:ken@roarmarketingconcepts.com', '#', '#', '#', '#', '#', '#', '#', '#', '#', '#', '#', '#', '#', '#', '#']


## First step: Have GPT-4o-mini figure out which links are relevant

### Use a call to gpt-4o-mini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [6]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}
"""

In [7]:
print(link_system_prompt)

You are provided with a list of links found on a webpage. You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}



In [8]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [9]:
print(get_links_user_prompt(ken))

Here is the list of links on the website of https://roarmarketingconcepts.github.io/ - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
#intro
#work
#resume
#social
https://github.com/ROARMarketingConcepts/AWS-Training/blob/main/AWS_Sagemaker_Models/PCA%20%26%20XGBoost%20Classification/cardiovascular_disease_prediction_notebook.ipynb
https://github.com/ROARMarketingConcepts/Recurrent-Neural-Network-Examples/blob/master/RNN_Training_on_Time_Series_Data_Example.ipynb
https://github.com/ROARMarketingConcepts/Marketing-Attribution-Using-Markov-Chains/blob/main/Marketing_Channel_Attribution_Using_Markov_Chains.ipynb
http://www.screencast.com/t/52LICFT7rxY
https://github.com/ROARMarketingConcepts/
https://rpubs.com/ROARMarketingConcepts/
https://drive.google.com/file/d/1so0HcugmQyg5EvGISEYzyeKD5zEsMwFa/view?usp=share

In [10]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [12]:
get_links("https://roarmarketingconcepts.github.io/")

{'links': [{'type': 'github page',
   'url': 'https://github.com/ROARMarketingConcepts/'},
  {'type': 'rpubs page', 'url': 'https://rpubs.com/ROARMarketingConcepts/'},
  {'type': 'linkedin page',
   'url': 'https://www.linkedin.com/in/kenwood-roarmarketingconcepts/'}]}

In [13]:
# Anthropic has made their site harder to scrape, so I'm using HuggingFace..

huggingface = Website("https://huggingface.co")
huggingface.links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/posts',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/ds4sd/SmolDocling-256M-preview',
 '/mistralai/Mistral-Small-3.1-24B-Instruct-2503',
 '/manycore-research/SpatialLM-Llama-1B',
 '/sesame/csm-1b',
 '/deepseek-ai/DeepSeek-V3-0324',
 '/models',
 '/spaces/stabilityai/stable-virtual-camera',
 '/spaces/Trudy/gemini-codrawing',
 '/spaces/sesame/csm-1b',
 '/spaces/ByteDance/InfiniteYou-FLUX',
 '/spaces/prs-eth/thera',
 '/spaces',
 '/datasets/nvidia/Llama-Nemotron-Post-Training-Dataset-v1',
 '/datasets/glaiveai/reasoning-v1-20m',
 '/datasets/nvidia/PhysicalAI-Robotics-GR00T-X-Embodiment-Sim',
 '/datasets/FreedomIntelligence/medical-o1-reasoning-SFT',
 '/datasets/Congliu/Chinese-DeepSeek-R1-Distill-data-110k',
 '/datasets',
 '/join',
 '/pricing#endpoints',
 '/pricing#spaces',
 '/pricing',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/alle

In [14]:
get_links("https://huggingface.co")

{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'discussion page', 'url': 'https://discuss.huggingface.co'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT4-o

In [15]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [17]:
print(get_all_details("https://roarmarketingconcepts.github.io/"))

Found links: {'links': [{'type': 'company page', 'url': 'https://github.com/ROARMarketingConcepts'}, {'type': 'portfolio page', 'url': 'https://rpubs.com/ROARMarketingConcepts'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/in/kenwood-roarmarketingconcepts/'}]}
Landing page:
Webpage Title:
Kenneth G. Wood - Data Scientist
Webpage Contents:
Kenneth G. Wood
Data Scientist
Welcome to my Website!
Intro
Work
Resume
Social
Intro
I am a seasoned data scientist who selects, trains and tunes machine learning algorithms (linear/logistic regression, classifiers, neural networks) to transform structured and unstructured data into actionable insights and 'stories'.  The results and value I produce typically drive and shape business strategies and product roadmaps for cloud-based and SaaS software technology solutions. I am proficient with a number of data analysis and programming tools such as Python (including Pandas, NumPy, SciKit-Learn & TensorFlow), R, SQL, Excel and Tableau.
Work

In [21]:
# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information."

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short humorous, entertaining, jokey brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."


In [22]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [25]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}]}


'You are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\nLanding page:\nWebpage Title:\nHugging Face – The AI community building the future.\nWebpage Contents:\nHugging Face\nModels\nDatasets\nSpaces\nPosts\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 1M+ models\nTrending on\nthis week\nModels\nds4sd/SmolDocling-256M-preview\nUpdated\n1 day ago\n•\n27.9k\n•\n852\nmistralai/Mistral-Small-3.1-24B-Instruct-2503\nUpdated\n2 days ago\n•\n60.4k\n•\n924\nmanycore-research/SpatialLM-Llama-1B\nUpdated\n4 days ago\n•\n2.38k\n•\n588\nsesame/csm-1b\nUpdated\n8 days ago\n•\n32k\n•\n1.58k\ndeepseek-ai/DeepSeek-V3-0324\nUpdated\nabout 7 hours ago\n•\n487\nBrowse 1M+ models\nSpaces\nRunning\non\nL40

In [23]:
get_brochure_user_prompt("Ken Wood", "https://roarmarketingconcepts.github.io/")

Found links: {'links': [{'type': 'about page', 'url': 'https://github.com/ROARMarketingConcepts/'}, {'type': 'social page', 'url': 'https://rpubs.com/ROARMarketingConcepts/'}, {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/kenwood-roarmarketingconcepts/'}]}


"You are looking at a company called: Ken Wood\nHere are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\nLanding page:\nWebpage Title:\nKenneth G. Wood - Data Scientist\nWebpage Contents:\nKenneth G. Wood\nData Scientist\nWelcome to my Website!\nIntro\nWork\nResume\nSocial\nIntro\nI am a seasoned data scientist who selects, trains and tunes machine learning algorithms (linear/logistic regression, classifiers, neural networks) to transform structured and unstructured data into actionable insights and 'stories'.  The results and value I produce typically drive and shape business strategies and product roadmaps for cloud-based and SaaS software technology solutions. I am proficient with a number of data analysis and programming tools such as Python (including Pandas, NumPy, SciKit-Learn & TensorFlow), R, SQL, Excel and Tableau.\nWork\nA machine learning project where I used AWS Sagemaker to perform prin

In [24]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [26]:
create_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'company page', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'community page', 'url': 'https://discuss.huggingface.co'}]}


# Welcome to Hugging Face: Where AI Hugs Back! 🤗

### The AI Community Building the Future

Join the ranks of over 50,000 organizations and become part of a vibrant community that delights in shaping tomorrow with AI! And we promise—it’s less awkward than trying to hug someone you don’t know.

---

## Why Hugging Face? 

### 1. **Cutting-Edge Models**
Tired of your old, boring AI models? Explore **1M+ jaw-dropping models** that were "hugged" into existence! From "SmolDocling-256M" to "DeepSeek-V3," these models are on more diets than most people—constantly updated to keep them trendy.

### 2. **Datasets Galore**
Access **250k+ datasets** so rich that they make Scrooge McDuck’s gold vault look like a piggy bank. Dive into datasets for every ML task imaginable (and some you didn’t know you needed)!

### 3. **Spaces for AI Thoughts**
Running an AI project? We’ve got **400k+ applications** where creativity and coding collide. Whether it's crafting conversational speech or enhancing images with real-time super-resolution, there’s a space bursting with possibilities. Who knew saving the world could feel like doodling in a digital art class?

---

## **Culture: We Hug! And Other Fun Stuff**
At Hugging Face, we believe in collaboration, creativity, and of course, the essence of a good hug! Our culture is defined by:
- **Community First:** We're all about breaking barriers (and maybe a few diets).
- **Open Source Happiness:** Sharing is caring, especially when the code is easy to give away and you don’t have to share your dessert.
- **Future-Forward:** If it’s not innovative, ditch it! We’re building the future, one hug at a time.

---

## Who’s Using Our Platform?
From the exclamation mark companies like **Meta, Amazon, Google, Intel, and Microsoft**—you know, just the usual suspects in the world of tech—everyone loves a good AI model! Just don’t ask Google for dating advice—it’s still figuring that out. 

---

## Careers: Join Our Hug Squad! 🤗
Looking for a career that embraces your passion for AI? Look no further! We’re on the lookout for innovators, dreamers, and code wizards! Positions ranging from model wranglers to dataset guardians available (capes optional). 

**Perks include:**
- An office environment that’s at least 10% friendlier than your average corporate gig.
- Opportunities to work with some of the brightest minds in AI (don’t worry, they hug too!).
- The chance to actually tell people you work in a space where ML has become child’s play. 

---

## **Conclusion: Let’s Hug It Out!**
Join us at Hugging Face, where AI is not just something you abbreviate as “AI” but also a community that hugs through tech innovation! 

### 🔗 Website: [huggingface.co](https://huggingface.co)
### 📮 Questions? Shoot us a message, but don’t forget to add a virtual hug. 🤗

Your future in AI is just one hug away!

In [27]:
create_brochure("Ken Wood", "https://roarmarketingconcepts.github.io")

Found links: {'links': [{'type': 'about page', 'url': 'https://github.com/ROARMarketingConcepts'}, {'type': 'social page', 'url': 'https://www.linkedin.com/in/kenwood-roarmarketingconcepts/'}]}


# Welcome to Ken Wood: The Data Wizard

![Ken Wood](https://example.com/ken-wood-image) (*Okay, maybe this isn’t a real image, but imagine it!*)

---

## Who Is Ken Wood?

Ah, Kenneth G. Wood! Your friendly neighborhood Data Scientist. With a Ph.D. in crunching numbers and a background that’s more colorful than a child’s crayon box, Ken takes complicated algorithms and turns them into stories that even your grandma could understand. Drive your business strategies with data insights so compelling, they might even push your coffee shop to offer espresso-infused brownies!

## What Does Ken Do?

- **Predicting Heart Disease**: While most folks fear their annual check-up, Ken fearlessly dives into heart disease prediction. Using PCA and XGBoost, he’s putting the “heart” in “heartfelt data analytics.”

- **RNNs & Time-Series Predictions**: Ken doesn’t just predict the weather; he predicts the future! Armed with a 100-node recurrent neural network and bucket loads of TensorFlow, he’s basically a data fortune teller.

- **Markov Chains**: When it comes to tracking user journeys, Ken just walks the walk—of probability! His marketing attribution models are so insightful, they could probably talk back.

- **SQL Wizardry**: Ken isn’t just good with tables; he’s mastered them! If you think crunching numbers in Excel is magic, wait until you see Ken wave his SQL wand!

*“I promise to make your data sing like a canary in a coal mine!”* - Ken 

---

## Cultured Ken

### Work Hard, (Data) Play Harder

At the Ken Wood headquarters, humor fuels our machine learning algorithms! You can expect intense brainstorming sessions mixed with spontaneous outbursts of dad jokes. 

**Core Values**:
- *Predicting Success* - because who needs a crystal ball when you have data?
- *Open-Source Spirit* - sharing is caring, and we care about your data!
- *Encouragement Over Competition* - the only thing we compete in is who can find the most obscure statistical reference.

### Join the Ken Team! 

Did someone say “careers”? Yes, yes we did! If you’re an aspiring data scientist, statistical wizard, or simply someone who can appreciate a good pun while sifting through Excel spreadsheets, we want you on our team!

- **Current Openings**:
  - Data Jedi (must have lightsaber skills)
  - Algorithm Architect (hard hat NOT required)
  - SQL Ninja (shuriken skills preferred but not required)

*“We offer competitive salaries, unlimited coffee, and the chance to decipher the universe’s most complex mysteries—like why the printer always jams!"*

## Get in Touch!

Want to predict your business’s future with a sprinkle of Ken's magic? Or perhaps you just want to send a virtual high-five? Reach out below!

- **LinkedIn**: [Ken on LinkedIn](https://linkedin.com)
- **GitHub**: [Code Secrets](https://github.com/kenwood)
- **R Programming Site**: Watch out, R Revolution!

---

**Ken Wood - Your quirky data magician, turning numbers into novelties!** ✨

---

*Disclaimer: All data is served with a side of humor. No baby pandas were harmed in the creation of this brochure.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [28]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [30]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/about'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'help page', 'url': 'https://huggingface.co/docs'}]}


# Hugging Face – The AI Community Building the Future

**Welcome to Hugging Face!** Where everyone gets a warm fuzzy feeling while building the future of AI! 🤗 

---

### **Why Join Us?**

1. **Community Collaboration**: We're like a family of propeller-hatted nerds, but instead of building trees forts, we’re building the future. Join over **50,000 organizations** who are already here, including the tech titans: Google, Amazon, and Microsoft! (Yes, they *really* do get along!)

2. **Model Mania**: Explore the **1 million+ models** we’ve cooked up. Need a model to help with just about anything? From text to image, video, audio, and even 3D? We got you covered! 

3. **Dataset Delight**: We have over **250K datasets** - it's like a buffet, and yes, you can help yourself!

4. **Machine Learning Playground**: With **400K+ applications**, you'll feel like a kid in a candy store but with much more sophisticated flavors!

---

### **Enterprising Vibes** 💼

We aim to give your team the **most advanced platform** to go on the AI adventure of a lifetime! Our offerings include:
- Enterprise-grade security - we lock up our data better than a squirrel guarding a stash of acorns.
- Single Sign-On - because passwords should be a thing of the past, right?
- Priority Support - for when you really need to save the world and can’t wait on hold.

Pricing starts at just **$20/user/month** because we know how to stay friendly with your wallet! 

---

### **Join Our Team!** 🚀

At Hugging Face, we're not just about crunching numbers and bits; we’re about crunching life’s little joys! Check out our [careers page](#) for open positions. Our team culture is built on:
- Creativity: We believe in out-of-the-box ideas (and sometimes even out-of-this-world ideas).
- Collaboration: Join forces with other brilliant minds. You bring the dreams, we bring the models!
- Flexibility: Working from home or the couch? Your call! Pajama Mondays are a thing here.

---

### **Join the Hugging Face Movement!** 🌈

Whether you want to explore AI applications, collaborate on projects, or join our vibrant community, Hugging Face is where it's at! 

So, grab your metaphorical hug and join us in reshaping the future. 

For inquiries, applications, and heartwarming hugs, check us out at:
- **Website**: [Hugging Face](https://huggingface.co)
- **Social Media**: Follow us on Twitter, LinkedIn, and Discord!

### **Get Ready to Hug Some AI! 🤗** 

Remember, the future is a friendly place when you’re with Hugging Face!

In [32]:
stream_brochure("Ken Wood", "https://roarmarketingconcepts.github.io")

Found links: {'links': [{'type': 'company page', 'url': 'https://github.com/ROARMarketingConcepts/'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/in/kenwood-roarmarketingconcepts/'}]}



# Welcome to Ken Wood: The Data Whisperer! 🦸‍♂️📊

## Meet Ken G. Wood: Interactive Data Scientist Extraordinaire!

Hello, lovely reader! You may be wondering, **"Who on earth is Ken Wood, and why do I need him in my life?"** Well, put on your best data analysis goggles because you’re about to dive into the fascinating world of **Actionable Insights**... Yes, that’s right, we’re talking about the Magic of Data!

### 🎩 Ken's Data Magic Tricks
- **Machine Learning Maestro**: Ken tunes algorithms like a rock star—whether it's linear regression or the mysterious neural networks, Ken has them all in his playlist.
- **Heartfelt Predictive Analysis**: Using AWS Sagemaker, Ken has a knack for predicting heart disease, indicating he can certainly put the "science" in data science (and maybe help your heart in the process)!
- **Time Travel for Data**: With a 100-node recurrent neural network, Ken is practically Doctor Who when it comes to making predictions on time-series data. Allons-y!

### 🎯 Projects that Pack a Punch!
Ken’s projects combine his love for **data** and **storytelling**:
1. **Marketing Channel Attribution Using Markov Chains**: Moving from one marketing event to another with the finesse of a dancing octopus! 🐙 
2. **Increasing Profits by Harnessing Data Analytics**: Because who doesn’t want to become the next Steve Jobs of data? 

### 🤝 Who We Help!
- **Startups and Enterprises**: Whether you’re a scrappy startup or a corporate giant, our data-driven insights will help shape your strategies. Just think of us as the fairy godmothers of SaaS!
- **Marketers**: Want to know why your email to grandma didn’t turn into a sale? Ken’s here to analyze the journey of every confused customer like an emotional therapist for browsers. 📈

### 👨‍💻 Work with Us!
Ready to be part of Ken's data-crazed adventures? We’re looking for:
- **Data enthusiasts**: If you can make sense of numbers and squeeze out stories from spreadsheets, we want to hear from you! 
- **Creativity loving Coders**: If you’re fluent in Python, R, SQL, and can rock Tableau like a pro, then come join the party!

### 🎉 Company Culture
Here at Ken Wood Inc., we believe that **data** and **fun** can co-exist. Meetings may involve:
- Calculate Your Coffee Consumption session
- Predictive Pizza Friday (the more pizzas, the more data)
- Annual “Data Dance-Off”, where the best visualizer is crowned as the ‘Queen/King of Data’! 👑🎶

### 🌐 Let's Connect!
Want to be part of this data revolution? 
- **LinkedIn**: [Let's Connect!](your-link-here)
- **GitHub**: [Check out Ken’s Code!](your-github-link-here)

### 📩 Interested in joining? 
Fill up your details below, and let’s transform those data dreams into actionable solutions!

| Name  | Email  | Role Interested In  |
|-------|--------|---------------------|
|       |        |                     |

So whether you want to work with a team that makes numbers shout and data dance, or if you're here to get some data magic spun for your business direction, you've just found your go-to wizard—**Ken Wood**!

---

*Disclaimer: No actual wands or magical spells were used in transforming data into stories.*


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>